<a href="https://colab.research.google.com/github/alinaatiq-research/3DDGD-GNN/blob/main/3DDGD_GNN_Implementation_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3DDGD + GNN Implementation for 3D Deepfake Detection


Instead of flattening mesh features for Mesh-MLP / TabTransformer only, this implementation converts each `.obj` mesh into a graph:

`3D face mesh (.obj) → vertices + faces → edge_index graph → GCN/GAT/GraphSAGE → real/fake prediction`


**Label convention used here**

- `0 = real`
- `1 = fake`

## 1. Install dependencies

This GNN implementation avoids TensorFlow/MediaPipe unless you explicitly crop faces again. It trains from `.obj` meshes and uses `trimesh` + `torch-geometric`.


In [ ]:

# Runtime → Change runtime type → GPU before running this cell.
!nvidia-smi

!pip install -q trimesh scikit-learn pandas matplotlib tqdm plotly
!pip install -q torch-geometric

# Optional: graph ops will work without torch-scatter, but PyG may warn that scatter can be faster.
# Do not force-install torch-scatter unless you know the exact wheel for your Colab torch/cuda version.


## 2. Mounting Google Drive and setting project paths

The notebook can train directly from your mesh folders. It does **not** require the original model weights.


In [ ]:
from google.colab import drive
import os

# Mount Google Drive to the default mount point
drive.mount('/content/drive')

# Define the full path for the project
project_dir = '/content/drive/MyDrive/3DDGD'

# Create the directory if it doesn't exist
# This ensures that when other cells try to access PROJECT_DIR, it exists.
os.makedirs(project_dir, exist_ok=True)

print(f"Google Drive mounted. Project directory '{project_dir}' is ready.")

In [ ]:

import os
import sys
import json
import math
import random
import warnings
from pathlib import Path
from typing import List, Tuple, Optional, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, WeightedRandomSampler

import trimesh

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_undirected
from torch_geometric.nn import (
    GCNConv,
    GATConv,
    SAGEConv,
    global_mean_pool,
    global_max_pool,
)

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Torch:', torch.__version__)


## 3. Configuration

In [ ]:
REAL_ROOTS = ["/content/drive/MyDrive/3DDGD/datasets/Facescape"]
FAKE_ROOTS = ["/content/drive/MyDrive/3DDGD/datasets/3D2M"]

In [ ]:

# =========================
# MAIN PATH CONFIGURATION
# =========================
PROJECT_DIR = Path('/content/drive/MyDrive/3DDGD')
DATASETS_DIR = PROJECT_DIR / 'datasets'

# IMPORTANT: adjust these paths according to your Drive structure.
# Anything under REAL_ROOTS will be labelled 0 = real.
# Anything under FAKE_ROOTS will be labelled 1 = fake.
REAL_ROOTS = [
    DATASETS_DIR / 'Facescape',          # common real source in 3DDGD
]

FAKE_ROOTS = [
    # DATASETS_DIR / 'DECA',               # if present
    # DATASETS_DIR / 'EMOCA',              # if present
    # DATASETS_DIR / 'fake',               # if present
    # DATASETS_DIR / 'Fake',               # if present
    DATASETS_DIR / '3D2M',             # uncomment ONLY if you intentionally use 3D2M as fake data
]

# If you already created a CSV with two columns: path,label then set it here.
# Leave as None to build labels from REAL_ROOTS and FAKE_ROOTS.
LABEL_CSV_PATH = None

# Output/cache folders
WORK_DIR = PROJECT_DIR / 'gnn_implementation'
GRAPH_CACHE_DIR = WORK_DIR / 'graph_cache'
MODEL_DIR = WORK_DIR / 'models'
RESULTS_DIR = WORK_DIR / 'results'
for p in [WORK_DIR, GRAPH_CACHE_DIR, MODEL_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Mesh preprocessing
MAX_FACES = 5000       # safer for Colab. Use 15000 for closer base-paper setting if GPU/RAM allows.
MIN_VERTICES = 50      # skip broken meshes
FORCE_REPROCESS = False
LIMIT_PER_CLASS = None # e.g. 200 for quick debugging; None for full dataset

# Training config
MODEL_TYPE = 'GCN'     # choose: 'GCN', 'GAT', or 'SAGE'
BATCH_SIZE = 2         # 2 or 4 for large meshes; increase if using MAX_FACES small
EPOCHS = 30
LR = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.25
HIDDEN_DIM = 128
NUM_LAYERS = 3
GAT_HEADS = 4
USE_WEIGHTED_SAMPLER = True  # helpful for imbalance such as many real and fewer fake samples

print('PROJECT_DIR:', PROJECT_DIR)
print('Work folder:', WORK_DIR)


In [ ]:
# from pathlib import Path
# import pandas as pd

# DATASETS_DIR = Path('/content/drive/MyDrive/3DDGD/datasets')

# obj_files = list(DATASETS_DIR.rglob('*.obj'))
# print("Total .obj files found:", len(obj_files))

# rows = []
# for p in obj_files:
#     rel = p.relative_to(DATASETS_DIR)
#     top_folder = rel.parts[0] if len(rel.parts) > 0 else ""
#     second_folder = rel.parts[1] if len(rel.parts) > 1 else ""
#     rows.append({
#         "top_folder": top_folder,
#         "second_folder": second_folder,
#         "full_path": str(p)
#     })

# scan_df = pd.DataFrame(rows)

# display(scan_df.groupby("top_folder").size().reset_index(name="obj_count").sort_values("obj_count", ascending=False))

# display(scan_df.head(20))

## 4. Build metadata: `.obj` paths + labels

In [ ]:

def find_obj_files(root: Path) -> List[Path]:
    root = Path(root)
    if not root.exists():
        return []
    return sorted([p for p in root.rglob('*.obj') if p.is_file()])


def build_metadata_from_roots(real_roots: List[Path], fake_roots: List[Path], limit_per_class=None) -> pd.DataFrame:
    real_files = []
    fake_files = []

    for root in real_roots:
        files = find_obj_files(root)
        print(f'REAL root: {root} -> {len(files)} .obj files')
        real_files.extend(files)

    for root in fake_roots:
        files = find_obj_files(root)
        print(f'FAKE root: {root} -> {len(files)} .obj files')
        fake_files.extend(files)

    # Remove duplicates while preserving order
    real_files = list(dict.fromkeys(map(str, real_files)))
    fake_files = list(dict.fromkeys(map(str, fake_files)))

    # Avoid overlap. If a path accidentally appears in both lists, remove it and show warning.
    overlap = set(real_files).intersection(set(fake_files))
    if overlap:
        print(f'WARNING: {len(overlap)} files appeared in both real and fake roots. Removing overlaps.')
        real_files = [p for p in real_files if p not in overlap]
        fake_files = [p for p in fake_files if p not in overlap]

    if limit_per_class is not None:
        real_files = real_files[:limit_per_class]
        fake_files = fake_files[:limit_per_class]

    rows = [{'path': p, 'label': 0, 'label_name': 'real'} for p in real_files]
    rows += [{'path': p, 'label': 1, 'label_name': 'fake'} for p in fake_files]

    meta = pd.DataFrame(rows)
    if len(meta) == 0:
        raise FileNotFoundError('No .obj files found. Check REAL_ROOTS and FAKE_ROOTS.')

    meta = meta.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return meta


if LABEL_CSV_PATH is not None:
    metadata = pd.read_csv(LABEL_CSV_PATH)
    assert {'path', 'label'}.issubset(metadata.columns), 'CSV must have path,label columns.'
    metadata['label'] = metadata['label'].astype(int)
    metadata['label_name'] = metadata['label'].map({0: 'real', 1: 'fake'})
else:
    metadata = build_metadata_from_roots(REAL_ROOTS, FAKE_ROOTS, LIMIT_PER_CLASS)

metadata_path = WORK_DIR / 'metadata.csv'
metadata.to_csv(metadata_path, index=False)

print('Saved metadata:', metadata_path)
display(metadata.head())
print(metadata['label_name'].value_counts())

assert metadata['label'].nunique() == 2, 'You need both real and fake samples before training.'


## 5. Mesh loading, normalization, and graph construction




In [ ]:

def load_trimesh_obj(path: str) -> Optional[trimesh.Trimesh]:
    '''Load an OBJ as a single Trimesh. Returns None if invalid.'''
    try:
        obj = trimesh.load(path, process=False, force='mesh')
        if isinstance(obj, trimesh.Scene):
            meshes = [g for g in obj.geometry.values() if isinstance(g, trimesh.Trimesh)]
            if not meshes:
                return None
            obj = trimesh.util.concatenate(meshes)
        if not isinstance(obj, trimesh.Trimesh):
            return None
        if obj.vertices is None or obj.faces is None:
            return None
        if len(obj.vertices) < MIN_VERTICES or len(obj.faces) < 10:
            return None
        # remove non-finite vertices/faces if possible
        obj.remove_unreferenced_vertices()
        return obj
    except Exception as e:
        return None


def simplify_mesh_if_needed(mesh: trimesh.Trimesh, max_faces: int = MAX_FACES) -> trimesh.Trimesh:
    '''Reduce face count for Colab memory. If simplification fails, returns original mesh.'''
    if max_faces is None or len(mesh.faces) <= max_faces:
        return mesh
    try:
        # trimesh versions differ in spelling of this function
        if hasattr(mesh, 'simplify_quadric_decimation'):
            return mesh.simplify_quadric_decimation(face_count=max_faces)
        if hasattr(mesh, 'simplify_quadratic_decimation'):
            return mesh.simplify_quadratic_decimation(max_faces)
    except Exception:
        pass
    return mesh


def normalize_vertices(vertices: np.ndarray) -> np.ndarray:
    vertices = vertices.astype(np.float32)
    vertices = np.nan_to_num(vertices, nan=0.0, posinf=0.0, neginf=0.0)
    center = vertices.mean(axis=0, keepdims=True)
    vertices = vertices - center
    scale = np.linalg.norm(vertices, axis=1).max()
    if scale < 1e-8:
        scale = 1.0
    vertices = vertices / scale
    return vertices.astype(np.float32)


def mesh_to_edge_index(faces: np.ndarray, num_nodes: int) -> torch.Tensor:
    faces = np.asarray(faces, dtype=np.int64)
    faces = faces[(faces >= 0).all(axis=1) & (faces < num_nodes).all(axis=1)]
    if len(faces) == 0:
        raise ValueError('No valid faces found after filtering.')

    face_t = torch.tensor(faces, dtype=torch.long)
    # triangle edges: (a,b), (b,c), (c,a)
    edges = torch.cat([
        face_t[:, [0, 1]],
        face_t[:, [1, 2]],
        face_t[:, [2, 0]],
    ], dim=0).t().contiguous()
    edge_index = to_undirected(edges, num_nodes=num_nodes)
    return edge_index


def build_node_features(mesh: trimesh.Trimesh, edge_index: torch.Tensor) -> torch.Tensor:
    vertices = normalize_vertices(np.asarray(mesh.vertices))
    num_nodes = len(vertices)

    try:
        normals = np.asarray(mesh.vertex_normals, dtype=np.float32)
        if normals.shape != vertices.shape:
            normals = np.zeros_like(vertices, dtype=np.float32)
    except Exception:
        normals = np.zeros_like(vertices, dtype=np.float32)
    normals = np.nan_to_num(normals, nan=0.0, posinf=0.0, neginf=0.0)

    # Degree is a simple topology feature.
    deg = np.bincount(edge_index.cpu().numpy().reshape(-1), minlength=num_nodes).reshape(-1, 1).astype(np.float32)
    deg = np.log1p(deg)
    deg = (deg - deg.mean()) / (deg.std() + 1e-6)

    # Radial distance and z-height are simple shape descriptors.
    radial = np.linalg.norm(vertices, axis=1, keepdims=True).astype(np.float32)
    z_height = vertices[:, 2:3].astype(np.float32)

    x = np.concatenate([vertices, normals, deg, radial, z_height], axis=1)  # 9 features
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return torch.tensor(x, dtype=torch.float32)


def graph_cache_file(mesh_path: str, label: int, max_faces: int) -> Path:
    safe_name = str(mesh_path).replace('/content/drive/MyDrive/', '').replace('/', '__').replace(' ', '_')
    safe_name = safe_name.replace('.obj', '')
    return GRAPH_CACHE_DIR / f'{safe_name}__label{label}__faces{max_faces}.pt'


def obj_to_pyg_graph(mesh_path: str, label: int, max_faces: int = MAX_FACES) -> Data:
    mesh = load_trimesh_obj(mesh_path)
    if mesh is None:
        raise ValueError('Could not load valid mesh.')

    mesh = simplify_mesh_if_needed(mesh, max_faces=max_faces)
    mesh.remove_unreferenced_vertices()

    num_nodes = len(mesh.vertices)
    if num_nodes < MIN_VERTICES:
        raise ValueError(f'Too few vertices: {num_nodes}')

    edge_index = mesh_to_edge_index(mesh.faces, num_nodes=num_nodes)
    x = build_node_features(mesh, edge_index)

    data = Data(
        x=x,
        edge_index=edge_index,
        y=torch.tensor([int(label)], dtype=torch.long),
        num_nodes=num_nodes,
    )
    data.path = str(mesh_path)
    data.num_faces = int(len(mesh.faces))
    return data


## 6. Converting all meshes


In [ ]:

def process_all_graphs(metadata: pd.DataFrame, force: bool = FORCE_REPROCESS) -> pd.DataFrame:
    processed_rows = []
    failed_rows = []

    for row in tqdm(metadata.itertuples(index=False), total=len(metadata), desc='Converting OBJ → graph'):
        mesh_path = str(row.path)
        label = int(row.label)
        cache_path = graph_cache_file(mesh_path, label, MAX_FACES)

        try:
            if force or not cache_path.exists():
                data = obj_to_pyg_graph(mesh_path, label, MAX_FACES)
                torch.save(data, cache_path)
            else:
                # check loadability
                try:
                    _ = torch.load(cache_path, map_location='cpu', weights_only=False)
                except TypeError:
                    _ = torch.load(cache_path, map_location='cpu')

            processed_rows.append({
                'path': mesh_path,
                'label': label,
                'label_name': 'fake' if label == 1 else 'real',
                'graph_path': str(cache_path),
            })
        except Exception as e:
            failed_rows.append({'path': mesh_path, 'label': label, 'error': repr(e)})

    processed = pd.DataFrame(processed_rows)
    failed = pd.DataFrame(failed_rows)

    processed.to_csv(WORK_DIR / 'processed_graphs.csv', index=False)
    failed.to_csv(WORK_DIR / 'failed_graphs.csv', index=False)

    print('Processed graphs:', len(processed))
    print('Failed graphs:', len(failed))
    if len(processed):
        print(processed['label_name'].value_counts())
    if len(failed):
        print('Sample failed rows:')
        display(failed.head())

    assert len(processed) > 0, 'No graphs were processed. Check your OBJ files.'
    assert processed['label'].nunique() == 2, 'Processed graphs must contain both real and fake classes.'
    return processed

processed_df = process_all_graphs(metadata, FORCE_REPROCESS)
processed_df.head()


## 7. Dataset and train/test split

This notebook uses a stratified 80/20 split, matching the general experimental style of the base paper.


In [ ]:

class CachedGraphDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        path = self.frame.loc[idx, 'graph_path']
        try:
            data = torch.load(path, map_location='cpu', weights_only=False)
        except TypeError:
            data = torch.load(path, map_location='cpu')
        return data


def make_split(df: pd.DataFrame, test_size: float = 0.20):
    labels = df['label'].astype(int).values
    min_count = df['label'].value_counts().min()
    stratify = labels if min_count >= 2 else None
    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=SEED,
        stratify=stratify,
    )
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    return train_df, test_df

train_df, test_df = make_split(processed_df, test_size=0.20)

print('Train counts:')
display(train_df['label_name'].value_counts())
print('Test counts:')
display(test_df['label_name'].value_counts())

train_dataset = CachedGraphDataset(train_df)
test_dataset = CachedGraphDataset(test_df)

# Infer input feature dimension from first graph
sample_graph = train_dataset[0]
IN_CHANNELS = sample_graph.x.shape[1]
print('Input node features:', IN_CHANNELS)
print('Sample graph:', sample_graph)


## 8. DataLoaders with class imbalance handling

In [ ]:

def make_weighted_sampler(frame: pd.DataFrame):
    labels = frame['label'].astype(int).values
    class_counts = np.bincount(labels, minlength=2)
    class_counts = np.maximum(class_counts, 1)
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[labels]
    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True,
    )
    return sampler

if USE_WEIGHTED_SAMPLER:
    train_sampler = make_weighted_sampler(train_df)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler)
else:
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print('Train batches:', len(train_loader))
print('Test batches:', len(test_loader))


## 9. GNN model: GAT


In [ ]:

class MeshGNN(nn.Module):
    def __init__(
        self,
        in_channels: int,
        hidden_dim: int = 128,
        num_layers: int = 3,
        model_type: str = 'GCN',
        dropout: float = 0.25,
        gat_heads: int = 4,
    ):
        super().__init__()
        self.model_type = model_type.upper()
        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        current_dim = in_channels
        for layer_idx in range(num_layers):
            if self.model_type == 'GCN':
                conv = GCNConv(current_dim, hidden_dim)
                out_dim = hidden_dim
            elif self.model_type == 'GAT':
                assert hidden_dim % gat_heads == 0, 'hidden_dim must be divisible by gat_heads for GAT.'
                conv = GATConv(
                    current_dim,
                    hidden_dim // gat_heads,
                    heads=gat_heads,
                    concat=True,
                    dropout=dropout,
                )
                out_dim = hidden_dim
            elif self.model_type == 'SAGE':
                conv = SAGEConv(current_dim, hidden_dim)
                out_dim = hidden_dim
            else:
                raise ValueError("MODEL_TYPE must be 'GCN', 'GAT', or 'SAGE'.")

            self.convs.append(conv)
            self.norms.append(nn.LayerNorm(out_dim))
            current_dim = out_dim

        # mean pool + max pool = 2 * hidden_dim graph embedding
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv, norm in zip(self.convs, self.norms):
            x = conv(x, edge_index)
            x = norm(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        pooled = torch.cat([
            global_mean_pool(x, batch),
            global_max_pool(x, batch),
        ], dim=1)
        logits = self.classifier(pooled).view(-1)
        return logits


model = MeshGNN(
    in_channels=IN_CHANNELS,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    model_type=MODEL_TYPE,
    dropout=DROPOUT,
    gat_heads=GAT_HEADS,
).to(DEVICE)

print(model)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))


## 10. Training and evaluation functions

Metrics reported:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- Confusion matrix


In [ ]:

def get_pos_weight(frame: pd.DataFrame) -> torch.Tensor:
    counts = frame['label'].value_counts().to_dict()
    neg = counts.get(0, 0)  # real
    pos = counts.get(1, 0)  # fake
    if pos == 0:
        return torch.tensor([1.0], dtype=torch.float32, device=DEVICE)
    return torch.tensor([max(neg / pos, 1.0)], dtype=torch.float32, device=DEVICE)


def evaluate(model, loader, threshold: float = 0.5) -> Dict[str, float]:
    model.eval()
    all_probs, all_labels = [], []
    total_loss = 0.0
    criterion_eval = nn.BCEWithLogitsLoss()

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            y = batch.y.float().view(-1)
            logits = model(batch)
            loss = criterion_eval(logits, y)
            probs = torch.sigmoid(logits)

            total_loss += float(loss.item()) * y.size(0)
            all_probs.extend(probs.detach().cpu().numpy().tolist())
            all_labels.extend(y.detach().cpu().numpy().astype(int).tolist())

    y_true = np.array(all_labels).astype(int)
    y_prob = np.array(all_probs).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        'loss': total_loss / max(len(y_true), 1),
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
    }
    if len(np.unique(y_true)) == 2:
        metrics['roc_auc'] = roc_auc_score(y_true, y_prob)
    else:
        metrics['roc_auc'] = float('nan')

    metrics['y_true'] = y_true
    metrics['y_pred'] = y_pred
    metrics['y_prob'] = y_prob
    return metrics


def train_one_epoch(model, loader, optimizer, criterion) -> float:
    model.train()
    running_loss = 0.0
    total = 0

    for batch in loader:
        batch = batch.to(DEVICE)
        y = batch.y.float().view(-1)

        optimizer.zero_grad(set_to_none=True)
        logits = model(batch)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        running_loss += float(loss.item()) * y.size(0)
        total += y.size(0)

    return running_loss / max(total, 1)


## 11. Train the GNN model

The best checkpoint is saved by validation/test F1-score.


In [ ]:

pos_weight = get_pos_weight(train_df)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)

print('pos_weight for fake class:', float(pos_weight.item()))

best_f1 = -1.0
best_path = MODEL_DIR / f'best_{MODEL_TYPE}_gnn.pt'
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    train_metrics = evaluate(model, train_loader)
    test_metrics = evaluate(model, test_loader)
    scheduler.step(test_metrics['f1'])

    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'train_acc': train_metrics['accuracy'],
        'train_f1': train_metrics['f1'],
        'test_loss': test_metrics['loss'],
        'test_acc': test_metrics['accuracy'],
        'test_precision': test_metrics['precision'],
        'test_recall': test_metrics['recall'],
        'test_f1': test_metrics['f1'],
        'test_roc_auc': test_metrics['roc_auc'],
        'lr': optimizer.param_groups[0]['lr'],
    }
    history.append(row)

    if test_metrics['f1'] > best_f1:
        best_f1 = test_metrics['f1']
        torch.save({
            'model_state_dict': model.state_dict(),
            'config': {
                'in_channels': IN_CHANNELS,
                'hidden_dim': HIDDEN_DIM,
                'num_layers': NUM_LAYERS,
                'model_type': MODEL_TYPE,
                'dropout': DROPOUT,
                'gat_heads': GAT_HEADS,
                'max_faces': MAX_FACES,
                'label_map': {'real': 0, 'fake': 1},
            },
            'best_f1': best_f1,
        }, best_path)

    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} train_f1={train_metrics['f1']:.4f} | "
        f"test_acc={test_metrics['accuracy']:.4f} test_f1={test_metrics['f1']:.4f} "
        f"auc={test_metrics['roc_auc']:.4f}"
    )

hist_df = pd.DataFrame(history)
hist_path = RESULTS_DIR / f'history_{MODEL_TYPE}.csv'
hist_df.to_csv(hist_path, index=False)
print('Best F1:', best_f1)
print('Saved best model:', best_path)
print('Saved history:', hist_path)


## 12. Plot training curves


In [ ]:

plt.figure(figsize=(8, 5))
plt.plot(hist_df['epoch'], hist_df['train_f1'], label='Train F1')
plt.plot(hist_df['epoch'], hist_df['test_f1'], label='Test F1')
plt.xlabel('Epoch')
plt.ylabel('F1-score')
plt.title(f'{MODEL_TYPE} GNN F1-score')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(hist_df['epoch'], hist_df['train_loss'], label='Train Loss')
plt.plot(hist_df['epoch'], hist_df['test_loss'], label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title(f'{MODEL_TYPE} GNN Loss')
plt.legend()
plt.grid(True)
plt.show()


## 13. Final evaluation using the best checkpoint


In [ ]:

def load_best_model(checkpoint_path: Path) -> MeshGNN:
    ckpt = torch.load(checkpoint_path, map_location=DEVICE)
    cfg = ckpt['config']
    loaded_model = MeshGNN(
        in_channels=cfg['in_channels'],
        hidden_dim=cfg['hidden_dim'],
        num_layers=cfg['num_layers'],
        model_type=cfg['model_type'],
        dropout=cfg['dropout'],
        gat_heads=cfg['gat_heads'],
    ).to(DEVICE)
    loaded_model.load_state_dict(ckpt['model_state_dict'])
    loaded_model.eval()
    return loaded_model

best_model = load_best_model(best_path)
final_metrics = evaluate(best_model, test_loader)

print('Final test metrics')
for k in ['loss', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    print(f'{k}: {final_metrics[k]:.4f}')

print('\nConfusion matrix [rows=true, cols=pred], label order: 0=real, 1=fake')
cm = confusion_matrix(final_metrics['y_true'], final_metrics['y_pred'], labels=[0, 1])
print(cm)

print('\nClassification report')
print(classification_report(
    final_metrics['y_true'],
    final_metrics['y_pred'],
    labels=[0, 1],
    target_names=['real', 'fake'],
    zero_division=0,
))

# Save predictions
pred_df = test_df.copy()
pred_df['prob_fake'] = final_metrics['y_prob']
pred_df['pred_label'] = final_metrics['y_pred']
pred_df['pred_name'] = pred_df['pred_label'].map({0: 'real', 1: 'fake'})
pred_path = RESULTS_DIR / f'predictions_{MODEL_TYPE}.csv'
pred_df.to_csv(pred_path, index=False)
print('Saved predictions:', pred_path)


## 14. Single OBJ inference with your trained GNN

Use this after training. It predicts one mesh as real/fake.


In [ ]:

def predict_single_obj(mesh_path: str, checkpoint_path: Path = best_path, threshold: float = 0.5):
    ckpt = torch.load(checkpoint_path, map_location=DEVICE)
    cfg = ckpt['config']

    single_model = MeshGNN(
        in_channels=cfg['in_channels'],
        hidden_dim=cfg['hidden_dim'],
        num_layers=cfg['num_layers'],
        model_type=cfg['model_type'],
        dropout=cfg['dropout'],
        gat_heads=cfg['gat_heads'],
    ).to(DEVICE)
    single_model.load_state_dict(ckpt['model_state_dict'])
    single_model.eval()

    # Label is dummy here because inference only needs x/edge_index.
    data = obj_to_pyg_graph(mesh_path, label=0, max_faces=cfg.get('max_faces', MAX_FACES))
    data.batch = torch.zeros(data.num_nodes, dtype=torch.long)
    data = data.to(DEVICE)

    with torch.no_grad():
        logit = single_model(data)
        prob_fake = torch.sigmoid(logit).item()

    pred_label = 1 if prob_fake >= threshold else 0
    pred_name = 'fake' if pred_label == 1 else 'real'
    return {'mesh_path': mesh_path, 'prob_fake': prob_fake, 'prediction': pred_name}

# Example: replace this with any .obj path you want to test.
# single_result = predict_single_obj('/content/drive/MyDrive/3DDGD/datasets/3D2M/FemaleFace1/FemaleFace1.obj')
# single_result


In [ ]:

def add_vertex_noise_to_data(data: Data, sigma: float = 0.005) -> Data:
    data2 = data.clone()
    # first 3 node features are normalized xyz coordinates
    noise = torch.randn_like(data2.x[:, :3]) * sigma
    data2.x[:, :3] = data2.x[:, :3] + noise
    return data2

class NoisyWrapper(Dataset):
    def __init__(self, base_dataset: Dataset, sigma: float):
        self.base_dataset = base_dataset
        self.sigma = sigma
    def __len__(self):
        return len(self.base_dataset)
    def __getitem__(self, idx):
        return add_vertex_noise_to_data(self.base_dataset[idx], sigma=self.sigma)

for sigma in [0.001, 0.003, 0.005, 0.01]:
    noisy_loader = DataLoader(NoisyWrapper(test_dataset, sigma=sigma), batch_size=BATCH_SIZE, shuffle=False)
    noisy_metrics = evaluate(best_model, noisy_loader)
    print(f'sigma={sigma:.3f} | acc={noisy_metrics["accuracy"]:.4f} f1={noisy_metrics["f1"]:.4f} auc={noisy_metrics["roc_auc"]:.4f}')


In [ ]:

summary = {
    'model_type': MODEL_TYPE,
    'max_faces': MAX_FACES,
    'num_graphs': len(processed_df),
    'train_real': int((train_df['label'] == 0).sum()),
    'train_fake': int((train_df['label'] == 1).sum()),
    'test_real': int((test_df['label'] == 0).sum()),
    'test_fake': int((test_df['label'] == 1).sum()),
    'best_f1': float(best_f1),
    'final_accuracy': float(final_metrics['accuracy']),
    'final_precision': float(final_metrics['precision']),
    'final_recall': float(final_metrics['recall']),
    'final_f1': float(final_metrics['f1']),
    'final_roc_auc': float(final_metrics['roc_auc']) if not math.isnan(final_metrics['roc_auc']) else None,
    'best_model_path': str(best_path),
}

summary_path = RESULTS_DIR / f'summary_{MODEL_TYPE}.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print('Saved summary:', summary_path)
